In [45]:
import pandas as pd
import numpy as np
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
submission_df = pd.read_csv('sample_submission.csv')
pd.set_option('display.max_columns', None)

In [ ]:
def preprocess_spaceship_data(input_df):
    target_df = input_df.copy()
    
    # ① PassengerId の分解
    target_df['Group_ID'] = target_df['PassengerId'].str.split('_').str[0]
    group_counts = target_df['Group_ID'].value_counts()
    target_df['True_Group_Size'] = target_df['Group_ID'].map(group_counts)
    
    # ② Cabin の分解
    cabin_split = target_df['Cabin'].astype(str).str.split('/')
    target_df['Cabin_Deck'] = cabin_split.str[0]
    target_df['Cabin_Side'] = cabin_split.str[2]
    target_df.loc[target_df['Cabin'].isnull(), ['Cabin_Deck', 'Cabin_Side']] = np.nan
    
    # 🌟 ③ 鈴木さん流：ソートしてグループ内コピペ穴埋め
    target_df = target_df.sort_values('Group_ID').reset_index(drop=True)
    
    # SideとDeckをそれぞれグループ内で上下からコピペ
    target_df['Cabin_Side'] = target_df.groupby('Group_ID')['Cabin_Side'].ffill().bfill().fillna('Unknown')
    target_df['Cabin_Deck'] = target_df.groupby('Group_ID')['Cabin_Deck'].ffill().bfill().fillna('Unknown')
    # 4. 【最終保険】グループに誰もいなくて、どうしても残ってしまったNullは 'Unknown' で埋める
    target_df['Cabin_Side'] = target_df['Cabin_Side'].fillna('Unknown')
    # 🌟 新しい作戦：船内での総消費額（Total_Spend）を計算
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
        # 各列のNullを0に置き換えてから、すべて横方向に足し算（axis=1）します
    target_df['Total_Spend'] = target_df[spend_cols].fillna(0).sum(axis=1)
    # プランBの書き方（1円でも使ってたら1、そうじゃなければ0）
    target_df['Is_Spender'] = np.where(target_df['Total_Spend'] > 0, 1, 0)
    return target_df

# 実行
processed_train = preprocess_spaceship_data(train_df)
processed_test  = preprocess_spaceship_data(test_df)

In [47]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
import numpy as np

# 1. 不要な列を落とし、文字列をカテゴリ型に変える
# ※鈴木さんが前処理で作った 'Group_ID' も、予測には使わないので drop_cols に追加しています
drop_cols = ['PassengerId', 'Cabin', 'Name', 'Group_ID']
X = processed_train.drop(columns=drop_cols + ['Transported'])
y = processed_train['Transported'].astype(int) # True/False を 1/0 に変換

# 文字列の列を自動で一括 category 型に
obj_cols = X.select_dtypes(include=['object']).columns
X[obj_cols] = X[obj_cols].astype('category')

# 2. 5分割のクロスバリデーションの準備
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🌟 【変更点】確率（小数の配列）を保存する空の箱を用意
oof_preds_lgb_proba = np.zeros(len(X))

print("--- LightGBM クロスバリデーション（確率版）開始 ---")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    model = LGBMClassifier(random_state=42, n_estimators=100, learning_rate=0.05)
    model.fit(X_train, y_train)
    
    # 🌟 【変更点】predict ではなく predict_proba を使う！
    # ※ [:, 1] をつけることで「1（転送された）である確率」だけを綺麗に抜き出せます
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    # 確率を箱に保存
    oof_preds_lgb_proba[val_idx] = y_pred_proba
    
    # 手元のスコア確認用（正解率を出すために、確率は0.5で一度0か1に変換して評価します）
    score = accuracy_score(y[val_idx], np.where(y_pred_proba > 0.5, 1, 0))
    print(f"Fold {fold+1} の正解率: {score:.5f}")

--- LightGBM クロスバリデーション（確率版）開始 ---
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000811 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1645
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
Fold 1 の正解率: 0.80966
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000479 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1645
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 15
[LightGBM] [Info] [binary:BoostFro

In [50]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import numpy as np

# 1. 不要な列を落とす
drop_cols = ['PassengerId', 'Cabin', 'Name', 'Group_ID']
X = processed_train.drop(columns=drop_cols + ['Transported'])

# 【バグ対策1】yを完全にNumPyの整数配列にする
y = processed_train['Transported'].astype(int).values

# 🌟【バグ対策2：今回の主犯】すべての文字列・カテゴリ列を、一度確実にstr型にしてからcategory型にする
for col in X.select_dtypes(include=['object', 'category']).columns:
    X[col] = X[col].astype(str).astype('category')

# 🌟 【変更点】XGBoost用の確率を保存する空の箱
oof_preds_xgb_proba = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    model_xgb = XGBClassifier(random_state=42, n_estimators=100, learning_rate=0.05, enable_categorical=True)
    model_xgb.fit(X_train, y_train)
    
    # 確率で予測を出す
    y_pred_proba_xgb = model_xgb.predict_proba(X_val)[:, 1]
    oof_preds_xgb_proba[val_idx] = y_pred_proba_xgb

# 🌟 全部のFoldが終わった後に、全体のスコアを計算
xgb_actual_score = accuracy_score(y, np.where(oof_preds_xgb_proba > 0.5, 1, 0))
print(f"✨ XGBoostの真のCVスコア: {xgb_actual_score:.5f}")

✨ XGBoostの真のCVスコア: 0.80444


In [51]:
# 💡 2つのAIが弾き出した「確率」の平均をとる（対等な多数決）
oof_proba_avg = (oof_preds_lgb_proba + oof_preds_xgb_proba) / 2

# 最終的に、平均確率が 0.5 を超えたら 1（転送）、以下なら 0（無事）とする
ensemble_preds = np.where(oof_proba_avg > 0.5, 1, 0)

# 究極の最終スコアを計算# 確率の平均をとる
oof_proba_avg = (oof_preds_lgb_proba + oof_preds_xgb_proba) / 2

# 0.5以上なら1、以下なら0
ensemble_preds = np.where(oof_proba_avg > 0.5, 1, 0)

# 手元データでの、アンサンブルの最終実力を計算！
final_cv_score = accuracy_score(y, ensemble_preds)
print(f"🏆 手元で検証したアンサンブルのCVスコア: {final_cv_score:.5f}")
final_score = accuracy_score(y, ensemble_preds)
print("--------------------------------")
print(f"🏆 確率アンサンブル完了！最終CVスコア: {final_score:.5f}")

🏆 手元で検証したアンサンブルのCVスコア: 0.80582
--------------------------------
🏆 確率アンサンブル完了！最終CVスコア: 0.80582


In [52]:
# -----------------------------------------------------------------
# 🌟 提出用データの予測フェーズ
# -----------------------------------------------------------------
# 1. テストデータ（X_test）を訓練データ（X）と全く同じ列、同じ型に揃える
X_test = processed_test.drop(columns=drop_cols)
X_test[obj_cols] = X_test[obj_cols].astype(str).astype('category')

# 2. 本番用として、訓練データ「全員分」を使ってLightGBMとXGBoostを再学習させる
# (クロスバリデーションではなく、データ全体をフルに使って最後の本気モデルを作ります)
final_model_lgb = LGBMClassifier(random_state=42, n_estimators=100, learning_rate=0.05)
final_model_lgb.fit(X, y)

final_model_xgb = XGBClassifier(random_state=42, n_estimators=100, learning_rate=0.05, enable_categorical=True)
final_model_xgb.fit(X, y)

# 3. テストデータの「確率」をそれぞれ予測する
test_preds_lgb_proba = final_model_lgb.predict_proba(X_test)[:, 1]
test_preds_xgb_proba = final_model_xgb.predict_proba(X_test)[:, 1]

# 4. 2人の確率を平均してアンサンブル！
test_proba_avg = (test_preds_lgb_proba + test_preds_xgb_proba) / 2

# 5. 確率が0.5を超えたら 1(True)、以下なら 0(False) にする
test_ensemble_preds = np.where(test_proba_avg > 0.5, 1, 0)

# -----------------------------------------------------------------
# 📂 CSVファイルの作成
# -----------------------------------------------------------------
# Kaggleの指定ルールに従って、PassengerId と Transported の形にする
submission = pd.DataFrame({
    'PassengerId': processed_test['PassengerId'],
    'Transported': test_ensemble_preds.astype(bool) # 1/0 を True/False に戻す
})

# CSVとして保存（インデックス番号は不要なので index=False）
submission.to_csv('submission_final.csv', index=False)
print("🏆 提出用ファイル 'submission_final.csv' が完成しました！")

[LightGBM] [Info] Number of positive: 4378, number of negative: 4315
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000869 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1650
[LightGBM] [Info] Number of data points in the train set: 8693, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503624 -> initscore=0.014495
[LightGBM] [Info] Start training from score 0.014495
🏆 提出用ファイル 'submission_final.csv' が完成しました！


In [ ]:
#Optunaを使うとき用
#import optuna

# 1. Optunaに「この箱（関数）の中のスコアを最大にして！」と頼む
def objective(trial):
    # 🌟 ここがパズル！Optunaに「この範囲で探してきて」と指定する
    suggested_n_estimators = trial.suggest_int('n_estimators', 50, 300)
    suggested_lr = trial.suggest_float('learning_rate', 0.01, 0.1)
    
    # 鈴木さんのクロスバリデーションコード（コピペ）
    # モデルの定義部分に、Optunaが選んだ数字を入れて学習させる
    model = LGBMClassifier(
        n_estimators=suggested_n_estimators, 
        learning_rate=suggested_lr,
        random_state=42
    )
    
    # 〜（中略：5分割して予測する処理）〜
    
    # 最終的な手元のCVスコアをOptunaに返してあげる
    return final_cv_score

# 2. あとは「100回実験して、一番良い組み合わせを見つけて！」と命令するだけ
study = optuna.create_study(direction='maximize') # スコアを最大化したい
study.optimize(objective, n_trials=100)           # 100回自動ループ

print(f"🏆 最高スコアの組み合わせ: {study.best_params}")